# PDF Preprocessing
This notebook prepares handwritten PDFs for OCR.

import os

REPO_PATH = "/content/Handwritten-Notebook-anlaysis---Automated-feedback"
os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import os
import sys

try:
    import numpy as np
    if int(np.__version__.split('.')[0]) >= 2:
        print("Downgrading NumPy to 1.x...")
        !pip install "numpy<2" -q
        print("NumPy downgraded. Restarting Colab runtime automatically...")
        os.kill(os.getpid(), 9) # Automatically restarts the Colab kernel
    else:
        print(f"NumPy version is {np.__version__} (Compatible 1.x)")
except ImportError:
    !pip install "numpy<2" -q
    os.kill(os.getpid(), 9) 

Downgrading NumPy to 1.x...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatibl

: 

: 

: 

In [ ]:
!pip install pymupdf==1.24.0 -q
!pip install opencv-python-headless==4.9.0.80 -q
!pip install Pillow==10.3.0 -q
print("Libraries ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 37.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.8/30.8 MB 58.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 14.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 41.3 MB/s eta 0:00:0000:0100:01
Libraries ready


In [ ]:
import fitz
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

print("All imports successful")

All imports successful


In [1]:
from google.colab import files

print("Upload a scanned handwritten notebook PDF to test the pipeline")
uploaded = files.upload()

PDF_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {PDF_PATH}")

Upload a scanned handwritten notebook PDF to test the pipeline


KeyboardInterrupt: 

In [ ]:
def pdf_to_images(pdf_path: str, dpi: int = 200) -> list:
    """Convert each PDF page to a raw RGB image at specified DPI."""
    doc = fitz.open(pdf_path)
    images = []
    
    print(f"PDF has {len(doc)} page(s)")
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        # Scale matrix for DPI (72 is PDF default DPI)
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        
        # Convert pixmap to numpy array
        img_array = np.frombuffer(pix.samples, dtype=np.uint8)
        img = img_array.reshape(pix.height, pix.width, pix.n)
        
        # Handle RGBA (4 channels) to RGB (3 channels)
        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
        
        images.append(img)
        print(f"  Page {page_num + 1}: {pix.width}x{pix.height} pixels")
    
    doc.close()
    return images

# Run PDF to images
raw_images = pdf_to_images(PDF_PATH, dpi=200)
print(f"\nTotal pages extracted: {len(raw_images)}")

NameError: name 'PDF_PATH' is not defined

In [ ]:
num_pages = min(2, len(raw_images))
fig, axes = plt.subplots(1, num_pages, figsize=(14, 10))

if num_pages == 1:
    axes = [axes]

for i in range(num_pages):
    axes[i].imshow(raw_images[i])
    axes[i].set_title(f'Raw Page {i+1} (before preprocessing)')
    axes[i].axis('off')

plt.suptitle('Raw PDF Pages', fontsize=14)
plt.tight_layout()
plt.show()

NameError: name 'raw_images' is not defined

In [ ]:
def to_grayscale(img: np.ndarray) -> np.ndarray:
    """Convert RGB image to grayscale."""
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

# Test on page 1
gray = to_grayscale(raw_images[0])

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(raw_images[0])
axes[0].set_title('Original RGB')
axes[0].axis('off')
axes[1].imshow(gray, cmap='gray')
axes[1].set_title('After Grayscale')
axes[1].axis('off')
plt.tight_layout()
plt.show()

NameError: name 'np' is not defined

In [ ]:
def remove_ruled_lines(gray: np.ndarray) -> np.ndarray:
    """Remove horizontal ruled lines from notebook pages."""
    # Create horizontal structuring element
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    
    # Detect horizontal lines
    detected_lines = cv2.morphologyEx(
        gray, cv2.MORPH_OPEN, horizontal_kernel, iterations=2
    )
    
    # Find contours of lines and remove them
    cnts = cv2.findContours(
        detected_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    
    result = gray.copy()
    for c in cnts:
        cv2.drawContours(result, [c], -1, (255, 255, 255), 2)
    
    return result

# Test line removal
no_lines = remove_ruled_lines(gray)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Before Line Removal')
axes[0].axis('off')
axes[1].imshow(no_lines, cmap='gray')
axes[1].set_title('After Line Removal')
axes[1].axis('off')
plt.tight_layout()
plt.show()

NameError: name 'np' is not defined

In [ ]:
def denoise(img: np.ndarray) -> np.ndarray:
    """Remove noise from grayscale image."""
    return cv2.fastNlMeansDenoising(img, h=10, templateWindowSize=7, searchWindowSize=21)

# Test denoising
denoised = denoise(no_lines)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(no_lines, cmap='gray')
axes[0].set_title('Before Denoising')
axes[0].axis('off')
axes[1].imshow(denoised, cmap='gray')
axes[1].set_title('After Denoising')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def deskew(img: np.ndarray) -> np.ndarray:
    """Correct page tilt using Hough transform."""
    coords = np.column_stack(np.where(img < 200))
    
    if len(coords) == 0:
        return img
    
    angle = cv2.minAreaRect(coords)[-1]
    
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    
    # Only correct small tilts (less than 10 degrees)
    if abs(angle) > 10:
        print(f"Skew angle {angle:.2f}° too large, skipping correction")
        return img
    
    print(f"Correcting skew angle: {angle:.2f}°")
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed = cv2.warpAffine(
        img, M, (w, h),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_REPLICATE
    )
    return deskewed

# Test deskewing
deskewed = deskew(denoised)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(denoised, cmap='gray')
axes[0].set_title('Before Deskewing')
axes[0].axis('off')
axes[1].imshow(deskewed, cmap='gray')
axes[1].set_title('After Deskewing')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def binarize(img: np.ndarray) -> np.ndarray:
    """Apply Otsu thresholding to produce clean black and white image."""
    _, binary = cv2.threshold(
        img, 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )
    return binary

# Test binarization
binary = binarize(deskewed)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(deskewed, cmap='gray')
axes[0].set_title('Before Binarization')
axes[0].axis('off')
axes[1].imshow(binary, cmap='gray')
axes[1].set_title('After Binarization (Final)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def preprocess_image(img: np.ndarray) -> np.ndarray:
    """Complete preprocessing pipeline for one page image."""
    gray = to_grayscale(img)
    no_lines = remove_ruled_lines(gray)
    denoised = denoise(no_lines)
    deskewed = deskew(denoised)
    binary = binarize(deskewed)
    return binary

def process_pdf(pdf_path: str) -> list:
    """Full pipeline: PDF to list of preprocessed images."""
    raw_images = pdf_to_images(pdf_path)
    processed = []
    for i, img in enumerate(raw_images):
        print(f"Preprocessing page {i+1}...")
        processed.append(preprocess_image(img))
    print(f"\nAll {len(processed)} pages preprocessed successfully")
    return processed

# Run full pipeline
processed_images = process_pdf(PDF_PATH)

In [ ]:
num_pages = min(2, len(processed_images))
fig, axes = plt.subplots(num_pages, 2, figsize=(14, 10 * num_pages))

if num_pages == 1:
    axes = [axes]

for i in range(num_pages):
    axes[i][0].imshow(raw_images[i])
    axes[i][0].set_title(f'Page {i+1} — Original')
    axes[i][0].axis('off')
    
    axes[i][1].imshow(processed_images[i], cmap='gray')
    axes[i][1].set_title(f'Page {i+1} — After Full Preprocessing')
    axes[i][1].axis('off')

plt.suptitle('Raw vs Preprocessed Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
preprocessing_code = '''import fitz
import cv2
import numpy as np
from PIL import Image
import os

def pdf_to_images(pdf_path: str, dpi: int = 200) -> list:
    doc = fitz.open(pdf_path)
    images = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat)
        img_array = np.frombuffer(pix.samples, dtype=np.uint8)
        img = img_array.reshape(pix.height, pix.width, pix.n)
        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
        images.append(img)
    doc.close()
    return images

def to_grayscale(img: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

def remove_ruled_lines(gray: np.ndarray) -> np.ndarray:
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    detected_lines = cv2.morphologyEx(gray, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
    cnts = cv2.findContours(detected_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    result = gray.copy()
    for c in cnts:
        cv2.drawContours(result, [c], -1, (255, 255, 255), 2)
    return result

def denoise(img: np.ndarray) -> np.ndarray:
    return cv2.fastNlMeansDenoising(img, h=10, templateWindowSize=7, searchWindowSize=21)

def deskew(img: np.ndarray) -> np.ndarray:
    coords = np.column_stack(np.where(img < 200))
    if len(coords) == 0:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) > 10:
        return img
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def binarize(img: np.ndarray) -> np.ndarray:
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary

def preprocess_image(img: np.ndarray) -> np.ndarray:
    gray = to_grayscale(img)
    no_lines = remove_ruled_lines(gray)
    denoised = denoise(no_lines)
    deskewed = deskew(denoised)
    binary = binarize(deskewed)
    return binary

def process_pdf(pdf_path: str) -> list:
    raw_images = pdf_to_images(pdf_path)
    processed = [preprocess_image(img) for img in raw_images]
    print(f"Preprocessing complete: {len(processed)} pages ready for OCR")
    return processed
'''

with open('src/preprocessing.py', 'w') as f:
    f.write(preprocessing_code)

print("src/preprocessing.py saved")

In [ ]:
TOKEN = "your_actual_token_here"
USERNAME = "Ritesh6767"
REPO = "Handwritten-Notebook-anlaysis---Automated-feedback"

!git add .
!git commit -m "Notebook 02 complete: PDF preprocessing pipeline with PyMuPDF and OpenCV"
!git remote set-url origin https://{TOKEN}@github.com/{USERNAME}/{REPO}.git
!git push origin main
print("Notebook 02 pushed to GitHub")